# 01 · Tu propio partido

1. Consigue el vídeo en `.mp4` (retransmisión con la **cámara principal**, la que sigue el juego desde
   la grada lateral; 720p o 1080p).
2. Ejecuta la instalación, sube el vídeo y ajusta los parámetros.
3. **Empieza siempre con un tramo corto** (`max_seconds=60`) y revisa el vídeo de verificación antes de
   procesar el partido entero.

Tiempos orientativos por minuto de vídeo a 25 fps: Colab GPU T4 ≈ 1 min, Mac M1/M2 ≈ 2–3 min,
CPU ≈ 30 min. Los vídeos a 50/60 fps se analizan automáticamente a ~25 fotogramas por segundo; para ir
aún más rápido en un partido completo usa `stride` = el doble de lo automático (la salida sigue a 10 fps).

**Qué se descarta a propósito** (como hace SkillCorner): primeros planos de jugadores o entrenadores,
planos del público o del banquillo, repeticiones desde otras cámaras. Ahí no se generan posiciones y el
vídeo de verificación lo indica con un aviso. Los entrenadores, jueces de línea y suplentes que aparecen
fuera de las líneas **no** se cuentan como jugadores.

In [ ]:
# 1) INSTALACIÓN — en Colab tarda ~2 min; en tu Mac (ya instalado) no hace nada.
# Descarga SIEMPRE la última versión del código (volver a ejecutar esta celda actualiza).
import sys, subprocess, os
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAMA = "claude/soccernet-calibration-pkg-r8517j"
    if not os.path.exists("tracking"):
        r = subprocess.run(["git", "clone", "-q", "-b", RAMA, "https://github.com/delioguzmang-maker/tracking"])
        if r.returncode != 0:  # la rama ya se fusionó: rama por defecto
            subprocess.run(["git", "clone", "-q", "https://github.com/delioguzmang-maker/tracking"], check=True)
    else:  # ya estaba descargado: traer lo último (si no, seguirías usando la versión vieja)
        subprocess.run(["git", "-C", "tracking", "fetch", "-q", "origin"], check=True)
        ref = "origin/" + RAMA
        if subprocess.run(["git", "-C", "tracking", "rev-parse", "-q", "--verify", ref], capture_output=True).returncode:
            ref = "origin/HEAD"
        subprocess.run(["git", "-C", "tracking", "reset", "-q", "--hard", ref], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tracking[notebooks]"], check=True)
    sys.path.insert(0, os.path.abspath("tracking"))
for _m in [m for m in sys.modules if m == "soccercal" or m.startswith("soccercal.")]:
    del sys.modules[_m]  # sin reiniciar el entorno: se carga el código recién descargado
import soccercal
print("soccercal", soccercal.version(), "listo")

## Elegir el vídeo

In [ ]:
VIDEO = "mi_partido.mp4"     # en tu Mac: ruta completa, p. ej. "/Users/tu_nombre/Downloads/partido.mp4"

if EN_COLAB and not os.path.exists(VIDEO):
    # Opción A: subir desde el ordenador (lento para archivos grandes)
    from google.colab import files
    subido = files.upload()
    VIDEO = next(iter(subido))
    # Opción B (recomendada para partidos): Google Drive
    # from google.colab import drive; drive.mount("/content/drive")
    # VIDEO = "/content/drive/MyDrive/partido.mp4"
print(VIDEO)

## Parámetros
| parámetro | qué hace |
|---|---|
| `start_s`, `max_seconds` | tramo a procesar (segundos) |
| `stride` | `None` = automático (~25 fotogramas analizados por segundo); 2 = la mitad de eso |
| `jersey_ocr` | leer dorsales (la etiqueta pasa de `id12` a `#17` cuando se lee con seguridad) |
| `jersey_views` | vistas de cada jugador que se leen en la segunda mirada (80; menos = más rápido) |
| `ball_refine` | segunda búsqueda del balón en alta resolución donde no se encontró (`True`) |
| `players_per_team` | máximo de identidades por equipo (11 = 10 de campo + portero; `None` = sin límite) |
| `det_model` | `yolo11m.pt` (equilibrado), `yolo11s.pt` (rápido), `yolo11x.pt` (preciso) |
| `home_name`, `away_name` | nombres de los equipos (el "local" es el primer color detectado; revisa `summary.json`) |
| `period`, `time_offset_s` | parte del partido y minuto de reloj del primer fotograma |
| `extrapolate` | rellenar jugadores fuera de cámara (`is_detected=False`), como SkillCorner |

In [ ]:
import soccercal
from soccercal import Config
cfg = Config(
    start_s=0, max_seconds=60,      # <- primero 60 s; luego None para todo
    stride=None,                    # automático
    det_model="yolo11m.pt",
    jersey_ocr=True,                # leer dorsales
    home_name="Local", away_name="Visitante",
    period=1, time_offset_s=0,
)
res = soccercal.run(VIDEO, "salida_partido", cfg)
res.summary()

## Revisa la calidad
* `pct_frames_with_camera`: fotogramas con calibración válida. Los primeros planos, repeticiones y
  cámaras detrás de la portería se descartan a propósito (SkillCorner hace lo mismo).
* `median_line_alignment`: 0,6–0,8 es bueno; < 0,4 indica un vídeo difícil (líneas poco visibles).
* Mira `salida_partido/verificacion.mp4`: las líneas magenta deben caer sobre las líneas del campo.

In [ ]:
import pandas as pd
cams = pd.read_csv("salida_partido/cameras.csv")
ax = cams.plot(x="t", y="line_alignment", style=".", figsize=(12, 3), title="calidad de calibración (keyframes)")
cams["valid"].mean()

## Si los equipos salen cambiados
El "equipo A" es el primer grupo de color. Para cambiar nombres basta con volver a ejecutar solo la
parte rápida (sin repetir las redes neuronales):

In [ ]:
from soccercal.pipeline import build
from soccercal.analysis import Analysis
an = Analysis.load("salida_partido/analysis.pkl.gz")
cfg.home_name, cfg.away_name = cfg.away_name, cfg.home_name
res = build(an, cfg)
res.save("salida_partido")
res.summary()